# Huấn luyện phân vùng đường cho camera xe thật
Chạy các ô từ trên xuống. Ô cấu hình là nơi đổi dữ liệu và kiến trúc. `FAKE_DATA=True` kiểm tra quy trình trên CPU (bộ xử lý trung tâm); chỉ số từ dữ liệu giả không phản ánh chất lượng trên xe.

Tập train (huấn luyện) và val (đánh giá) tách theo vị trí. Nhãn PNG là chỉ số lớp, không phải ảnh màu. ONNX (định dạng trao đổi mạng nơ-ron) nhận RGB, NCHW (lô ảnh, kênh màu, cao, rộng), float32 0–1. Tiền xử lý nằm ngoài mạng. Đầu ra logits (điểm chưa đổi thành xác suất) dùng sigmoid riêng cho mỗi lớp.


In [ ]:
TASK = "road"
PROJECT = "ten-project-duong"
WORKSPACE = None
VERSION = 1
INPUT_HW = (96,160)  # (cao, rộng); có thể chọn (128,224).
ARCH = "unet_b_minus"  # "nanounet" hoặc "unet_b_minus".
CLASS_NAMES = ["road"]  # Hoặc ["lane"], hoặc nhiều lớp tiền cảnh.
NUM_CLASSES = 1
STEPS = 3000
BATCH = 8
LR = 0.001
SEED = 42
FAKE_DATA = True
FAKE_STEPS = 2
INSTALL_DEPS = True
LOCAL_ROOT = "./MyDrive/UIT_Car_Racing/Vong3"
VAL_RATIO = 0.2
VAL_GROUPS = []  # Ví dụ ["target_3"] hoặc ["road_l4"].
SCAN_RATIOS = [0.60,0.70,0.80]  # Tỉ lệ chiều cao, không phải pixel cố định.
BRIGHTNESS = [0.6,0.8,1.0,1.2,1.4]
MASK_THRESHOLD = 0.5


In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if INSTALL_DEPS:
    packages = ["torch", "torchvision", "numpy<3", "opencv-python-headless", "PyYAML", "onnx>=1.14,<2", "onnxruntime>=1.16,<2", "thop", "matplotlib", "roboflow", "requests"]
    if TASK == "yolo":
        packages += ["ultralytics==8.4.161", "onnxslim"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/UIT_Car_Racing/Vong3")
else:
    ROOT = Path(LOCAL_ROOT).expanduser().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
# Mọi dữ liệu, bộ nhớ đệm và kết quả đều ở Vong3.
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / "cache" / "ultralytics")
os.environ["TORCH_HOME"] = str(ROOT / "cache" / "torch")
os.environ["MPLCONFIGDIR"] = str(ROOT / "cache" / "matplotlib")
import re, csv, json, random, shutil, hashlib, datetime, getpass
import numpy as np
import cv2, yaml, torch
import torch.nn as nn
import torch.nn.functional as F
import onnx, onnxruntime as ort
from thop import profile
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2 if FAKE_DATA else max(1, min(8, os.cpu_count() or 1)))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = "cpu" if FAKE_DATA else ("cuda:0" if torch.cuda.is_available() else "cpu")
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN = ROOT / TASK / ("fake" if FAKE_DATA else "real") / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)
print("Thiết bị:", DEVICE, "| Kết quả:", RUN)

def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def write_csv(path, records, fields):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(records)

def rf_project():
    from roboflow import Roboflow
    key = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("ROBOFLOW_API_KEY")
        except Exception:
            pass
    if not key:
        key = getpass.getpass("Khoá Roboflow (không lưu): ")
    rf = Roboflow(api_key=key)
    del key
    return rf.workspace(WORKSPACE).project(PROJECT) if WORKSPACE else rf.workspace().project(PROJECT)

RF_SUFFIX = re.compile(r"_(?:jpg|jpeg|png|bmp)\.rf\.[A-Za-z0-9]+$", re.I)
NAME_RE = re.compile(r"^(?P<session>.+?)_p(?P<pass_id>\d+)_l(?P<luot>\d+)_t(?P<target>\d+)_(?P<frame>\d+)$")

def original_stem(path):
    return RF_SUFFIX.sub("", Path(path).stem)

def metadata(path):
    stem = original_stem(path)
    m = NAME_RE.fullmatch(stem)
    if not m:
        raise ValueError("Tên ảnh không có lượt/mục tiêu/pass: " + stem)
    d = m.groupdict()
    for k in ("pass_id", "luot", "target", "frame"):
        d[k] = int(d[k])
    d["stem"] = stem
    # Cùng mục tiêu ở các lượt/phiên phải ở cùng tập. Với t00 (đường liên tục),
    # chưa có vị trí cụ thể: giữ nguyên cả lượt, không chia từng khung hay pass.
    d["group"] = "target_{}".format(d["target"]) if d["target"] else "road_l{}".format(d["luot"])
    return d

def split_records(records):
    groups = sorted(set(r["group"] for r in records))
    if len(groups) < 2:
        raise ValueError("Cần ít nhất 2 vị trí; t00 cần ít nhất 2 lượt độc lập.")
    if VAL_GROUPS:
        vg = set(VAL_GROUPS)
        if not vg <= set(groups):
            raise ValueError("VAL_GROUPS chứa vị trí không tồn tại")
    else:
        shuffled = groups.copy(); random.Random(SEED).shuffle(shuffled)
        vg = set(shuffled[:max(1, min(len(groups)-1, round(len(groups)*VAL_RATIO)))])
    for r in records:
        r["split"] = "val" if r["group"] in vg else "train"
    train = [r for r in records if r["split"] == "train"]
    val = [r for r in records if r["split"] == "val"]
    assert train and val
    assert not set(r["group"] for r in train) & set(r["group"] for r in val)
    write_csv(RUN / "split.csv", [{k:r[k] for k in ("stem","session","luot","target","pass_id","group","split")} for r in records],
              ["stem","session","luot","target","pass_id","group","split"])
    print("Train:", len(train), "| Val:", len(val), "| Vị trí val:", sorted(vg))
    return train, val


## Đọc ảnh và nhãn
Chọn phiên bản Roboflow `png-mask-semantic`, không có tăng cường dữ liệu, cắt hay đổi tên làm mất vị trí. Đọc `_classes.csv` để lấy đúng giá trị điểm ảnh cho từng tên lớp; thiếu lớp sẽ dừng thay vì đoán. PNG dạng bảng màu được giữ nguyên chỉ số. Các phiên bản có mặt nạ RGB cần xuất lại dưới dạng mặt nạ chỉ số.


In [ ]:
assert len(CLASS_NAMES) == NUM_CLASSES and len(set(CLASS_NAMES)) == NUM_CLASSES
assert ARCH in ("nanounet","unet_b_minus")
H,W = INPUT_HW
assert H % 8 == 0 and W % 8 == 0 and min(H,W) >= 32
from PIL import Image
DATA = RUN / "dataset"; DATA.mkdir()
if FAKE_DATA:
    folder = DATA / "train"; folder.mkdir()
    with (folder / "_classes.csv").open("w",encoding="utf-8",newline="") as f:
        writer = csv.writer(f); writer.writerow(["Pixel Value","Class"])
        writer.writerow([0,"background"])
        for i,n in enumerate(CLASS_NAMES): writer.writerow([i+1,n])
    rng = np.random.default_rng(SEED)
    for i in range(24):
        stem = "fake_p{:03d}_l{}_t{:02d}_{:04d}".format(i//6+1,i%4+1,i%4+1,i)
        im = np.full((H,W,3),50,np.uint8); mask = np.zeros((H,W),np.uint8)
        for c in range(NUM_CLASSES):
            x0 = int(W*(0.15+0.55*c/max(1,NUM_CLASSES)))
            x1 = min(W-1,x0+max(2,W//(NUM_CLASSES+3)))
            cv2.rectangle(mask,(x0,H//3),(x1,H-1),c+1,-1)
        im[mask>0] = (130,140,150)
        im = np.clip(im.astype(float)+rng.normal(0,3,im.shape),0,255).astype(np.uint8)
        cv2.imwrite(str(folder/(stem+".jpg")),im)
        Image.fromarray(mask).save(folder/(stem+"_mask.png"))
else:
    project = rf_project()
    project.version(VERSION).download("png-mask-semantic",location=str(DATA),overwrite=True)

class_files = sorted(DATA.rglob("_classes.csv"))
if not class_files: raise ValueError("Thiếu _classes.csv")
value_maps = []
for p in class_files:
    with p.open(encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        cols = {k.strip().lower():k for k in reader.fieldnames}
        vc = next((v for k,v in cols.items() if "pixel" in k or "value" in k),None)
        cc = next((v for k,v in cols.items() if "class" in k),None)
        if not vc or not cc: raise ValueError("Không đọc được cột lớp: "+str(p))
        mapping = {r[cc].strip():int(r[vc]) for r in reader}
    if not set(CLASS_NAMES) <= set(mapping): raise ValueError("Thiếu lớp: "+str(p))
    value_maps.append({n:mapping[n] for n in CLASS_NAMES})
assert all(m==value_maps[0] for m in value_maps), "Bảng lớp giữa các tập không khớp"
CLASS_VALUES = value_maps[0]
records=[]; seen=set()
for p in sorted(DATA.rglob("*")):
    if p.suffix.lower() not in (".jpg",".jpeg",".png") or p.stem.endswith("_mask"): continue
    mp = p.with_name(p.stem+"_mask.png")
    if not mp.exists(): raise ValueError("Thiếu mask: "+str(p))
    r = metadata(p)
    if r["stem"] in seen: raise ValueError("Ảnh trùng tên gốc: "+r["stem"])
    seen.add(r["stem"])
    im=cv2.imread(str(p)); mask=np.array(Image.open(mp))
    if im is None or mask.ndim!=2 or mask.shape!=im.shape[:2]:
        raise ValueError("Ảnh/mặt nạ sai định dạng: "+str(p))
    r.update(image=p,mask=mp); records.append(r)
train,val = split_records(records)


In [ ]:
def camera_aug(image, rng):
    """Chỉ đổi màu/cường độ, giữ nguyên hình học để khung và mask còn đúng."""
    x = image.astype(np.float32)
    mean = x.mean(axis=(0,1), keepdims=True)
    x = (x-mean)*rng.uniform(0.8,1.2)+mean
    x *= rng.uniform(0.75,1.25)
    x *= rng.uniform(0.85,1.15,(1,1,3))  # Lệch cân bằng trắng.
    h,w = x.shape[:2]
    if rng.random() < 0.4:
        # Một vùng hầm tối và ám màu đèn, không phải làm tối toàn ảnh.
        x0 = int(rng.uniform(0,0.35)*w); x1 = int(rng.uniform(0.65,1)*w)
        y0 = int(rng.uniform(0,0.45)*h)
        tint = np.array([rng.uniform(0.6,1.0),rng.uniform(0.65,1.0),rng.uniform(0.85,1.2)])
        x[y0:,x0:x1] *= rng.uniform(0.12,0.4)*tint
    if rng.random() < 0.3:
        k = int(rng.choice([3,5,7])); kernel = np.zeros((k,k),np.float32)
        if rng.random() < 0.5:
            kernel[k//2,:] = 1/k
        else:
            kernel[:,k//2] = 1/k
        x = cv2.filter2D(x,-1,kernel)
    if rng.random() < 0.5:
        x += rng.normal(0,rng.uniform(1,9),x.shape)
    if rng.random() < 0.3:
        yy,xx = np.mgrid[:h,:w]
        stripe = xx - rng.uniform(0.1,0.9)*w - rng.uniform(-0.4,0.4)*yy
        glare = np.exp(-0.5*(stripe/max(1,w*rng.uniform(0.015,0.06)))**2)
        x += glare[...,None]*rng.uniform(50,160)
    return np.clip(x,0,255).astype(np.uint8)

# Lưu ảnh xem trước để đối chiếu tăng cường dữ liệu với ảnh gốc.
def preview_aug(image):
    rng = np.random.default_rng(SEED)
    tiles = [image]+[camera_aug(image,rng) for _ in range(5)]
    cv2.imwrite(str(RUN / "augmentation.jpg"), np.concatenate(tiles,axis=1))


## Kiến trúc và huấn luyện
Hai biến thể U-Net (mạng bộ mã hoá/giải mã có nối tắt) được định nghĩa ngay trong notebook: NanoUNet dùng 3 mức 16/32/64/128; UNet B- dùng 2 mức 8/16/32. Đây là các bản triển khai tự chứa để huấn luyện mới. Mã `lane_models.py` từng dùng trước đây không có trong kho hiện tại nên không bảo đảm tương thích trọng số cũ.

Mặc định UNet B- ở 96×160; đếm phép tính ngay trước huấn luyện. GFLOPs là tỉ phép toán, quy ước 1 MAC (phép nhân cộng) = 2 phép toán. Ước lượng 6 ms/GFLOP chỉ để sàng lọc; cần đo TensorRT FP16 trên Jetson. Huấn luyện tối thiểu hoá BCE (sai số nhị phân) và Dice (độ chồng lấp), giữ trọng số có IoU tốt nhất trên val.


In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self,a,b):
        super().__init__(nn.Conv2d(a,b,3,padding=1),nn.ReLU(inplace=False),
                         nn.Conv2d(b,b,3,padding=1),nn.ReLU(inplace=False))

class SmallUNet(nn.Module):
    def __init__(self,widths,nc):
        super().__init__()
        self.enc = nn.ModuleList([DoubleConv(a,b) for a,b in zip([3]+list(widths[:-1]),widths)])
        self.dec = nn.ModuleList([DoubleConv(widths[i+1]+widths[i],widths[i]) for i in reversed(range(len(widths)-1))])
        self.head = nn.Conv2d(widths[0],nc,1)
    def forward(self,x):
        skips=[]
        for i,block in enumerate(self.enc):
            if i: x=F.max_pool2d(x,2)
            x=block(x); skips.append(x)
        for block,skip in zip(self.dec,reversed(skips[:-1])):
            x=F.interpolate(x,size=skip.shape[-2:],mode="bilinear",align_corners=False)
            x=block(torch.cat([x,skip],dim=1))
        return self.head(x)

class NanoUNet(SmallUNet):
    def __init__(self,nc): super().__init__((16,32,64,128),nc)

class UNetBMinus(SmallUNet):
    def __init__(self,nc): super().__init__((8,16,32),nc)

model = (NanoUNet(NUM_CLASSES) if ARCH=="nanounet" else UNetBMinus(NUM_CLASSES)).to(DEVICE)
macs,params = profile(model,inputs=(torch.zeros(1,3,H,W,device=DEVICE),),verbose=False)
GFLOPS = float(2*macs/1e9); PARAMS = int(sum(p.numel() for p in model.parameters()))
print("Tham số:",PARAMS,"| GFLOPs:",GFLOPS,"| Nano ước lượng ms:",6*GFLOPS)
if ARCH=="unet_b_minus" and INPUT_HW==(96,160): assert GFLOPS*6<=20
if GFLOPS*6>20: print("Cấu hình vượt 20 ms ước lượng; nên giảm kích thước hoặc đổi kiến trúc.")

def read_pair(r,augment=False,rng=None):
    im=cv2.resize(cv2.imread(str(r["image"])),(W,H),interpolation=cv2.INTER_LINEAR)
    mask=np.array(Image.open(r["mask"]))
    mask=cv2.resize(mask,(W,H),interpolation=cv2.INTER_NEAREST)
    y=np.stack([(mask==CLASS_VALUES[n]).astype(np.float32) for n in CLASS_NAMES])
    if augment:
        im=camera_aug(im,rng)
        if rng.random()<0.5:
            im=im[:,::-1].copy(); y=y[:,:,::-1].copy()
    x=np.ascontiguousarray(im[:,:,::-1].transpose(2,0,1),dtype=np.float32)/255
    return torch.from_numpy(x),torch.from_numpy(y)

preview_aug(cv2.resize(cv2.imread(str(train[0]["image"])),(W,H)))
optimizer=torch.optim.Adam(model.parameters(),lr=LR)
rng=np.random.default_rng(SEED)
best=-1; history=[]
@torch.no_grad()
def val_iou():
    model.eval(); inter=np.zeros(NUM_CLASSES); union=np.zeros(NUM_CLASSES)
    for r in val:
        x,y=read_pair(r); pred=model(x[None].to(DEVICE)).sigmoid()[0].cpu().numpy()>=MASK_THRESHOLD
        gt=y.numpy()>0.5
        inter+=(pred & gt).sum(axis=(1,2)); union+=(pred|gt).sum(axis=(1,2))
    return float(np.mean(np.divide(inter,union,out=np.ones_like(inter),where=union>0)))
steps=FAKE_STEPS if FAKE_DATA else STEPS
for step in range(1,steps+1):
    model.train()
    pairs=[read_pair(train[int(i)],True,rng) for i in rng.integers(0,len(train),size=min(BATCH,2) if FAKE_DATA else BATCH)]
    x=torch.stack([p[0] for p in pairs]).to(DEVICE); y=torch.stack([p[1] for p in pairs]).to(DEVICE)
    logits=model(x); prob=logits.sigmoid()
    dice=1-((2*(prob*y).sum((2,3))+1)/(prob.sum((2,3))+y.sum((2,3))+1)).mean()
    loss=F.binary_cross_entropy_with_logits(logits,y)+dice
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step==1 or step%max(1,steps//20)==0 or step==steps:
        score=val_iou(); history.append(dict(step=step,loss=float(loss.item()),iou=score))
        if score>best:
            best=score
            torch.save({"state_dict":model.state_dict(),"arch":ARCH,"input_hw":INPUT_HW,"class_names":CLASS_NAMES},RUN/"road.pth")
        print(history[-1])
write_csv(RUN/"training.csv",history,["step","loss","iou"])
checkpoint=torch.load(RUN/"road.pth",map_location=DEVICE,weights_only=False)
model.load_state_dict(checkpoint["state_dict"]); model.eval()


## Đánh giá độ sáng và mép theo hàng quét
IoU (tỉ lệ giao trên hợp) tính trên toàn tập cho từng lớp. Mép trái/phải là điểm đầu/cuối của lớp trên hàng, sai số chia chiều rộng ảnh. Báo riêng hàng mất mép và hàng dự đoán thừa; các hàng không có nhãn không được tính thành sai số bằng 0. Với vạch làn, hai mép là biên ngoài cùng của nhãn trên hàng quét.


In [ ]:
@torch.no_grad()
def evaluate_road():
    result=[]
    for gain in BRIGHTNESS:
        inter=np.zeros(NUM_CLASSES); union=np.zeros(NUM_CLASSES)
        edge={ (c,s):[] for c in range(NUM_CLASSES) for s in SCAN_RATIOS }
        missing={k:0 for k in edge}; extra={k:0 for k in edge}
        for r in val:
            x,y=read_pair(r); gt=y.numpy()>0.5
            pred=model((x*gain).clamp(0,1)[None].to(DEVICE)).sigmoid()[0].cpu().numpy()>=MASK_THRESHOLD
            inter+=(pred & gt).sum((1,2)); union+=(pred|gt).sum((1,2))
            for c in range(NUM_CLASSES):
                for ratio in SCAN_RATIOS:
                    assert 0<=ratio<=1
                    row=min(H-1,round(ratio*(H-1))); k=(c,ratio)
                    a=np.flatnonzero(pred[c,row]); b=np.flatnonzero(gt[c,row])
                    if len(b) and len(a): edge[k].append((abs(int(a[0])-int(b[0]))+abs(int(a[-1])-int(b[-1])))/(2*W))
                    elif len(b): missing[k]+=1
                    elif len(a): extra[k]+=1
        for c,n in enumerate(CLASS_NAMES):
            for ratio in SCAN_RATIOS:
                k=(c,ratio)
                result.append(dict(brightness=gain,cls=n,iou=float(inter[c]/union[c]) if union[c] else None,
                                   row_ratio=ratio,edge_error_width=float(np.mean(edge[k])) if edge[k] else None,
                                   matched_rows=len(edge[k]),missing_rows=missing[k],extra_rows=extra[k]))
    return result
metrics=evaluate_road()
write_csv(RUN/"road_metrics.csv",metrics,["brightness","cls","iou","row_ratio","edge_error_width","matched_rows","missing_rows","extra_rows"])
print(*metrics,sep="\n")


## Xuất và đối chiếu ONNX
Dùng opset 12, đầu vào cố định, số thực FP32. Đối chiếu trên ít nhất 20 ảnh không tăng cường (dữ liệu giả nếu bật chế độ thử), sai số xác suất tối đa phải nhỏ hơn 0,001. Kết quả gồm `.pth`, `.onnx`, `_info.json`, CSV và ảnh tăng cường. Chưa tạo engine (bản thực thi tối ưu) tại đây; build FP16 (số thực 16 bit) bằng TensorRT trên chính Jetson.


In [ ]:
export_model=model.cpu().eval()
onnx_path=RUN/"road.onnx"
torch.onnx.export(export_model,torch.zeros(1,3,H,W),str(onnx_path),opset_version=12,
                  input_names=["images"],output_names=["logits"],do_constant_folding=True,dynamo=False)
graph=onnx.load(str(onnx_path)); onnx.checker.check_model(graph)
assert graph.opset_import[0].version==12
assert [d.dim_value for d in graph.graph.input[0].type.tensor_type.shape.dim]==[1,3,H,W]
session=ort.InferenceSession(str(onnx_path),providers=["CPUExecutionProvider"])
checks=[]
# Giữ ảnh val rồi bổ sung train chỉ để đối chiếu số học, không tính chỉ số chất lượng.
check_records=val+train
if len(check_records)<20 and not FAKE_DATA: raise ValueError("Cần ít nhất 20 ảnh thật để đối chiếu ONNX")
for r in check_records[:max(20,len(val))]:
    x,_=read_pair(r); x=x[None].numpy()
    with torch.no_grad(): a=export_model(torch.from_numpy(x)).sigmoid().numpy()
    raw=session.run(None,{"images":x})[0]
    b=1/(1+np.exp(-np.clip(raw,-80,80)))
    checks.append(dict(stem=r["stem"],max_probability_error=float(np.max(np.abs(a-b)))))
assert len(checks)>=20 and max(r["max_probability_error"] for r in checks)<1e-3
write_csv(RUN/"onnx_check.csv",checks,["stem","max_probability_error"])
info={"task":"segmentation","architecture":ARCH,"class_names":CLASS_NAMES,"num_classes":NUM_CLASSES,
      "mask_pixel_values":CLASS_VALUES,"input_name":"images","input_shape":[1,3,H,W],"input_hw":[H,W],
      "input_dtype":"float32","color_order":"RGB","layout":"NCHW","resize":"stretch_bilinear",
      "normalization":{"scale":1/255,"mean":[0,0,0],"std":[1,1,1]},"preprocessing_in_model":False,
      "output_name":"logits","output_shape":[1,NUM_CLASSES,H,W],"output_type":"logits",
      "activation":"sigmoid_per_class","threshold":MASK_THRESHOLD,"opset":12,"dynamic":False,
      "parameters":PARAMS,"gflops":GFLOPS,"flops_convention":"2 FLOPs per MAC","nano_estimated_ms":6*GFLOPS,
      "latency_measured":False,"seed":SEED,"fake_data":FAKE_DATA,"project":PROJECT,"version":VERSION,
      "scan_row_ratios":SCAN_RATIOS,"onnx_check_images":len(checks),"onnx_max_probability_error":max(r["max_probability_error"] for r in checks),
      "torch_version":torch.__version__,"onnxruntime_version":ort.__version__}
write_json(RUN/"road_info.json",info)
print("Đã lưu:",RUN)
print("Trên Jetson: /usr/src/tensorrt/bin/trtexec --onnx=road.onnx --saveEngine=road_fp16.engine --fp16 --workspace=1024")
